In [1]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.xception import Xception #to get pre-trained model Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer #for text tokenization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from keras.layers import concatenate
from tensorflow.keras.utils import plot_model
from tensorflow.keras.layers import add
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense#Keras to build our CNN and LSTM
from tensorflow.keras.layers import LSTM, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint

In [2]:
from keras.models import Model

def print_layer_info(model):
    for layer in model.layers:
        print(layer.name)
        print("Input shape: ", layer.input_shape)
        print("Output shape: ", layer.output_shape)
        print()

In [73]:
# Load dataset with meme images and descriptions
dataset = pd.read_csv("memes_dataset1.csv")

# Define classes and one-hot encode them
classes = ["distracted_boyfriend", "this_is_fine", "mocking_spongebob", "first_world_problems", "arthur_fist","surprised_pikachu","pointing_spiderman"]
# classes = ["Distracted boyfriend", "This is fine", "Mocking Spongebob", "First world problems", "Arthur fist"]

class_dict = dict(zip(classes, range(len(classes))))
print("Class dictionary: ", class_dict)

Class dictionary:  {'distracted_boyfriend': 0, 'this_is_fine': 1, 'mocking_spongebob': 2, 'first_world_problems': 3, 'arthur_fist': 4, 'surprised_pikachu': 5, 'pointing_spiderman': 6}


In [74]:
dataset

,Image,Description,Class
0,distracted-boyfriend.jpg,A man looks at another woman while his girlfri...,distracted_boyfriend
1,this-is-fine.jpg,"A dog sits in a room that's on fire, saying ""T...",this_is_fine
2,mocking-spongebob.jpg,A picture of Spongebob Squarepants with altern...,mocking_spongebob
3,first-world-problems.jpg,"A man holding a phone, with the caption ""My ph...",first_world_problems
4,arthur-fist.jpg,A cartoon character named Arthur clenches his ...,arthur_fist
5,distracted-boyfriend.jpg,A stock photo of a man walking with his girlfr...,distracted_boyfriend
6,this-is-fine.jpg,"When everything around you is falling apart, b...",this_is_fine
7,mocking-spongebob.jpg,When someone is being ridiculous and you mock ...,mocking_spongebob
8,first-world-problems.jpg,When you're upset about something trivial beca...,first_world_problems
9,first-world-problems.jpg,When you complain about something that's reall...,first_world_problems


In [75]:
y = dataset["Class"].apply(lambda x: class_dict[x]).values
y = to_categorical(y, num_classes=len(classes))

# Tokenize text descriptions
tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset["Description"])

In [76]:
y

array([[1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
 

In [44]:
sequences = tokenizer.texts_to_sequences(dataset["Description"])
sequences

[[2, 15, 37, 10, 38, 39, 40, 16, 41, 37, 10, 72, 73],
 [2, 74, 75, 11, 2, 76, 42, 17, 77, 78, 79, 12, 80],
 [2, 81, 8, 82, 83, 18, 84, 85, 7, 86, 87, 11, 6, 43],
 [2, 15, 88, 2, 44, 18, 6, 43, 45, 44, 89, 90, 91, 92, 4, 93, 45, 94],
 [2, 95, 96, 97, 98, 99, 16, 100],
 [2,
  101,
  102,
  8,
  2,
  15,
  46,
  18,
  16,
  41,
  40,
  103,
  16,
  104,
  4,
  105,
  10,
  38,
  39,
  46,
  11,
  6,
  106,
  107],
 [1, 22, 108, 3, 12, 109, 110, 13, 3, 47, 4, 111, 112],
 [1, 9, 12, 48, 113, 7, 3, 114, 19],
 [1, 5, 49, 23, 14, 115, 116, 3, 24, 2, 117, 118],
 [1, 3, 119, 23, 14, 42, 25, 50, 20, 120],
 [1, 5, 51, 4, 121, 22, 3, 26, 7, 27, 49, 1, 3, 28, 24, 52],
 [1, 5, 53, 51, 4, 122, 20, 6, 123, 124, 125, 54, 2, 126, 127],
 [1, 5, 11, 2, 55, 56, 13, 128, 54, 129, 50, 130, 3],
 [1, 5, 11, 131, 23, 132, 55, 133, 25, 29],
 [1, 3, 47, 4, 134, 17, 2, 135, 57, 58, 136, 22, 12, 137, 138],
 [1, 3, 139, 5, 140, 4, 141, 142, 13, 14, 30, 143, 59, 144],
 [1, 5, 11, 2, 145, 13, 28, 146, 147, 10, 31, 148]

In [45]:
# Pad sequences to a fixed length
max_length = max([len(seq) for seq in sequences])
X = pad_sequences(sequences, maxlen=max_length, padding="post")

In [46]:
max_length

24

In [41]:
X

array([[  2,  15,  37,  10,  38,  39,  40,  16,  41,  37,  10,  72,  73,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  2,  74,  75,  11,   2,  76,  42,  17,  77,  78,  79,  12,  80,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  2,  81,   8,  82,  83,  18,  84,  85,   7,  86,  87,  11,   6,
         43,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  2,  15,  88,   2,  44,  18,   6,  43,  45,  44,  89,  90,  91,
         92,   4,  93,  45,  94,   0,   0,   0,   0,   0,   0],
       [  2,  95,  96,  97,  98,  99,  16, 100,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  2, 101, 102,   8,   2,  15,  46,  18,  16,  41,  40, 103,  16,
        104,   4, 105,  10,  38,  39,  46,  11,   6, 106, 107],
       [  1,  22, 108,   3,  12, 109, 110,  13,   3,  47,   4, 111, 112,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
       [  1,   9,  12,  48, 113,   7,   3

In [10]:
descriptions=list(dataset['Description'])

In [11]:
descriptions

['A man looks at another woman while his girlfriend looks at him disapprovingly',
 'A dog sits in a room that\'s on fire, saying "This is fine"',
 'A picture of Spongebob Squarepants with alternating uppercase and lowercase letters in the caption',
 'A man holding a phone, with the caption "My phone charger isn\'t long enough to reach my bed"',
 'A cartoon character named Arthur clenches his fist',
 'A stock photo of a man walking with his girlfriend while turning his head to look at another woman walking in the opposite direction',
 'When everything around you is falling apart, but you try to remain calm.',
 'When someone is being ridiculous and you mock them.',
 "When you're upset about something trivial because you have a privileged life.",
 "When you complain about something that's really not that important.",
 "When you're used to getting everything you want and get upset when you can't have it.",
 "When you're so used to convenience that the smallest inconvenience feels like a ma

In [13]:
len(classes)

7

In [12]:
to_categorical(1, num_classes=len(classes))

array([0., 1., 0., 0., 0., 0., 0.], dtype=float32)

In [50]:
def define_model(num_classes, vocab_size, max_length):
    # Define input layer for meme class and one-hot encode the class
    class_input = Input(shape=(num_classes,))
    class_dense = Dense(units=256, activation='relu')(class_input)

    # Define input layer for text description
    text_input = Input(shape=(max_length,))
    emb = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(text_input)
    lstm = LSTM(units=256)(emb)

    # Merge the class and LSTM output
    decoder1 = concatenate([class_dense, lstm])
    decoder2 = Dense(units=256, activation='relu')(decoder1)
    output = Dense(units=vocab_size, activation='softmax')(decoder2)

    # Define the model
    model = Model(inputs=[class_input, text_input], outputs=output)

    # Compile the model
    model.compile(loss='categorical_crossentropy', optimizer='adam')

    # Print the model summary and save a visualization of the model architecture to a file
    print(model.summary())
    plot_model(model, to_file='model.png', show_shapes=True)

    return model


In [14]:
import numpy as np
def data_generator(descriptions, classes, tokenizer, max_length, batch_size):
    target=list()
    x=list()
    labels=list()
    n=0
    # loop for ever over images
    while True:
        for j in range(len(descriptions)):
            n+=1
            # retrieve the photo feature
            class1 = classes[j]

            # encode the sequence
            seq = tokenizer.texts_to_sequences([descriptions[j]])[0]
            # split one sequence into multiple X, y pairs
            for i in range(1, len(seq)):
                # split into input and output pair
                in_seq, out_seq = seq[:i], seq[i]
                # pad input sequence
                in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                # encode output sequence
                out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                target.append(out_seq)
                x.append(in_seq)
                y = to_categorical(class_dict[class1], num_classes=7)
                labels.append(y)
            # batch size reached
            if n==batch_size:
                yield ([np.array(labels), np.array(x)], np.array(target))  # should yeild (61, 5) (61, 24) (61, 166)
                X1, X2, y = list(), list(), list()
                n=0


In [15]:
y.shape

(35, 7)

In [26]:
#To check the shape of the input and output for your model
[l,x],Y = next(data_generator(dataset['Description'], dataset['Class'], tokenizer, max_length,5))
print(l.shape, x.shape, Y.shape) # should print (61, 5) (61, 24) (61, 166)


(61, 7) (61, 24) (61, 206)


In [52]:
from keras.models import Model

def print_layer_info(model):
    for layer in model.layers:
        print(layer.name)
        print("Input shape: ", layer.input_shape)
        print("Output shape: ", layer.output_shape)
        print()

print_layer_info(model)

input_12
Input shape:  [(None, 24)]
Output shape:  [(None, 24)]

input_11
Input shape:  [(None, 7)]
Output shape:  [(None, 7)]

embedding_5
Input shape:  (None, 24)
Output shape:  (None, 24, 256)

dense_13
Input shape:  (None, 7)
Output shape:  (None, 256)

lstm_5
Input shape:  (None, 24, 256)
Output shape:  (None, 256)

concatenate_4
Input shape:  [(None, 256), (None, 256)]
Output shape:  (None, 512)

dense_14
Input shape:  (None, 512)
Output shape:  (None, 256)

dense_15
Input shape:  (None, 256)
Output shape:  (None, 206)



In [20]:
num_classes = len(dataset['Class'].unique())
num_classes


7

In [51]:
# Define the captioning model
vocab_size = len(tokenizer.word_index) + 1
model = define_model(num_classes, vocab_size, max_length)

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_12 (InputLayer)          [(None, 24)]         0           []                               
                                                                                                  
 input_11 (InputLayer)          [(None, 7)]          0           []                               
                                                                                                  
 embedding_5 (Embedding)        (None, 24, 256)      52736       ['input_12[0][0]']               
                                                                                                  
 dense_13 (Dense)               (None, 256)          2048        ['input_11[0][0]']               
                                                                                            

In [36]:
print_layer_info(model)

input_10
Input shape:  [(None, 24)]
Output shape:  [(None, 24)]

input_9
Input shape:  [(None, 5)]
Output shape:  [(None, 5)]

embedding_4
Input shape:  (None, 24)
Output shape:  (None, 24, 256)

dense_10
Input shape:  (None, 5)
Output shape:  (None, 256)

lstm_4
Input shape:  (None, 24, 256)
Output shape:  (None, 256)

concatenate_3
Input shape:  [(None, 256), (None, 256)]
Output shape:  (None, 512)

dense_11
Input shape:  (None, 512)
Output shape:  (None, 256)

dense_12
Input shape:  (None, 256)
Output shape:  (None, 206)



In [37]:
vocab_size

206

In [29]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


In [53]:
# os.mkdir("models_v6")
model = define_model(num_classes,vocab_size, max_length)
# Train the model using the data generator
batch_size = 5
steps_per_epoch = len(descriptions) // batch_size
epochs=10

for i in range(epochs):
    print(i)
    train_generator = data_generator(dataset['Description'], dataset['Class'], tokenizer, max_length,batch_size)
    history = model.fit_generator(train_generator, epochs=10, steps_per_epoch=steps_per_epoch)

    # Save the trained model weights
    model.save("models_v6/model_" + str(i) + ".h5")


Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_14 (InputLayer)          [(None, 24)]         0           []                               
                                                                                                  
 input_13 (InputLayer)          [(None, 7)]          0           []                               
                                                                                                  
 embedding_6 (Embedding)        (None, 24, 256)      52736       ['input_14[0][0]']               
                                                                                                  
 dense_16 (Dense)               (None, 256)          2048        ['input_13[0][0]']               
                                                                                            

C:\Users\crkpn\OneDrive\Desktop\MongoDB\mongosh-1.5.4-win32-x64\bin\ipykernel_5660\3594390494.py:11: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history = model.fit_generator(train_generator, epochs=10, steps_per_epoch=steps_per_epoch)


7/7 [==============================] - 7s 119ms/step - loss: 5.2884
Epoch 2/10
7/7 [==============================] - 1s 135ms/step - loss: 5.1375
Epoch 3/10
7/7 [==============================] - 1s 149ms/step - loss: 4.8508
Epoch 4/10
7/7 [==============================] - 1s 157ms/step - loss: 4.5052
Epoch 5/10
7/7 [==============================] - 1s 184ms/step - loss: 4.0755
Epoch 6/10
7/7 [==============================] - 2s 262ms/step - loss: 3.5692
Epoch 7/10
7/7 [==============================] - 2s 267ms/step - loss: 2.9980
Epoch 8/10
7/7 [==============================] - 2s 282ms/step - loss: 2.4490
Epoch 9/10
7/7 [==============================] - 2s 285ms/step - loss: 1.9796
Epoch 10/10
7/7 [==============================] - 2s 282ms/step - loss: 1.5835
1
Epoch 1/10
7/7 [==============================] - 1s 137ms/step - loss: 1.2803
Epoch 2/10
7/7 [==============================] - 1s 167ms/step - loss: 1.0781
Epoch 3/10
7/7 [==============================] - 1s 166ms/s

In [33]:
vocab_size

206

In [54]:
print(l.shape, x.shape, Y.shape) # should print (61, 5) (61, 24) (61, 166)

(61, 7) (61, 24) (61, 206)


In [55]:
pred=model.predict([l[0:2],x[0:2]])[0].argmax()

1/1 [==============================] - 6s 6s/step


In [58]:
word_for_id(pred, tokenizer)

'man'

In [59]:
l[:1].shape

(1, 7)

In [60]:
sequence.shape

NameError: name 'sequence' is not defined

In [62]:
def get_img():
    img=input('Enter image name: '+str(class_dict.keys()))
    temp = np.zeros((1,num_classes))
    temp[0][class_dict[img]]=1
    return temp

In [64]:
np.zeros((1,7))

array([[0., 0., 0., 0., 0., 0., 0.]])

In [65]:
str(class_dict)

"{'Distracted boyfriend': 0, 'This is fine': 1, 'Mocking Spongebob': 2, 'First world problems': 3, 'Arthur fist': 4, 'Surprised Pikachu': 5, 'Pointing Spider-Man': 6}"

In [67]:
def generate_desc(img,tokenizer):
    in_text = 'start'
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        pred = model.predict([img,sequence], verbose=0)
        pred = np.argmax(pred)
        word = word_for_id(pred, tokenizer)
        if word is None:
            break
        in_text += ' ' + word
        if word == 'end':
            break
    return in_text

In [57]:
def word_for_id(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

In [69]:
img=get_img()
words=generate_desc(img,tokenizer).split()

In [72]:
for i in range(1,len(words)-1):
    if words[i]==words[i+1]:
        continue
    print(words[i],end=' ')

image of something the pokémon franchise a of a of in the same options 

In [2]:
dataset

NameError: name 'dataset' is not defined